## Implement RAG with the Claude Messages API

### Installing Utilities and Libraries

In [1]:
%pip install anthropic==0.120.2 python-dotenv==1.2.2 openai==2.38.0 qdrant-client==1.14.3

Looking in indexes: https://pypi.org/simple/

[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: /usr/local/opt/python@3.10/bin/python3.10 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


### Setting up the Environment

In [3]:
import os
from dotenv import load_dotenv

load_dotenv()

# load claude model configurations
claude_model_name = os.getenv("CLAUDE_MODEL_NAME")
claude_api_key = os.getenv("CLAUDE_API_KEY")

# load openai model configurations
openai_api_key = os.getenv("OPENAI_API_KEY")

# load qdrant model configurations
qdrant_api_key = os.getenv("QDRANT_API_KEY")

# Declare QDrant DB Configurations
QDRANT_URL = "https://731133a6-5b34-4bea-a857-363834c2fd43.us-west-1-0.aws.cloud.qdrant.io:6333"
COLLECTION_NAME = "carbonops_enterprise_knowledge"

### Creating the Anthropic Client

In [4]:
import anthropic

anthropic_client = anthropic.Anthropic(api_key = claude_api_key)

### Creating the OpenAI Client

In [5]:
from openai import OpenAI

openai_client = OpenAI(
    api_key = openai_api_key
)

### Create the Embedding Generator Function

In [6]:
def generate_vector_embeddings(user_query):

    response = openai_client.embeddings.create(
        model = "text-embedding-3-small",
        input = user_query
    )

    return response.data[0].embedding

### Create the Context Retrieval Function

In [7]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

def context_retrieval(user_query):

    # Generating Vector Embeddings for the User Query
    user_embeddings = generate_vector_embeddings(user_query)

    # Initialize Qdrant client
    qdrant_client = QdrantClient(
        url=QDRANT_URL, 
        api_key=qdrant_api_key,
    )

    # Throwing a request to QDrantDB to retrieve the top-matched content along with the vector embeddings
    search_result = qdrant_client.query_points(
        collection_name = COLLECTION_NAME,
        query = user_embeddings,
        with_vectors=True,
        with_payload=True,
        limit=2
    ).points

    # Retrieving the top text content that can be used to answer the user query
    supporting_text_content = search_result[0].payload.get("content")
    return supporting_text_content

### Summarize with the Claude Messages API

In [8]:
user_query = """Tell me something about CarbonOps Global Remote Work Policy"""

context = context_retrieval(user_query)

with anthropic_client.messages.stream(
    model=claude_model_name,
    system = "You are a helpful AI Chatbot",
    messages=[
        {
            "role": "user",
            "content":  f"""answer the user query using the provided supporting knowledge \n 
                                            user query: {user_query} \n
                                            supporting_text: {context}""",
        }
    ],
    max_tokens=10000
) as stream:
    for text in stream.text_stream:
        print(text, end="", flush=True)

/var/folders/qr/9_d4tppn2rlg_xxqp53j80w80000gp/T/ipykernel_67159/1004231091.py:10: UserWarning: Qdrant client version 1.14.3 is incompatible with server version 1.19.1. Major versions should match and minor version difference must not exceed 1. Set check_compatibility=False to skip version check.
  qdrant_client = QdrantClient(


# CarbonOps Global Remote Work & Flexibility Policy — Overview

Here's a summary of CarbonOps' proprietary remote work policy:

## Basic Details
- **Version:** 1.0
- **Effective Date:** January 12, 2025 (01/12/2025)
- **Issued by:** CarbonOps People & Culture
- **Classification:** Internal Use Only

## Purpose
The policy is designed to empower employees to work effectively from wherever suits them best, while fostering a culture built on **trust, productivity, and inclusion**.

## Core Culture Principles
The policy is anchored in five key principles:
1. **Empowerment** – Trusting employees to decide where/how they work best
2. **Clarity** – Setting clear expectations and support structures
3. **Inclusion** – Ensuring flexible work access is fair across all roles/locations
4. **Productivity** – Tying flexibility to high performance and real outcomes
5. **Security** – Safeguarding company/customer data regardless of location

## "Flex Mode" — The Core Framework
CarbonOps' signature worki